In [ ]:
#!pip install -q numpy
#!pip install -q transformers datasets evaluate accelerate wandb

In [5]:
# If in Colab, then import the drive module from google.colab
if 'google.colab' in str(get_ipython()):
  # !pip install uv
  !pip install numpy -U -qq
  !pip install transformers evaluate wandb datasets accelerate trl peft bitsandbytes -U -qq
  !pip uninstall tensorflow -y -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 123.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.3.5 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.5 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [1]:
import sys
print(sys.version)


3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [2]:
import trl, transformers
print("trl:", trl.__version__)
print("transformers:", transformers.__version__)

trl: 0.26.1
transformers: 4.57.3


In [3]:
# standard python libraries
from pathlib import Path
from typing import Dict, List, Union, Optional, Tuple
from tqdm import tqdm
import json
import joblib
import os
import sys

# Data Science librraies
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import multilabel_confusion_matrix, precision_score, recall_score, f1_score

# Pytorch
import torch
import torch.nn as nn

# Huggingface Librraies
import evaluate
from datasets import load_dataset, DatasetDict, Dataset, ClassLabel
from trl import SFTConfig, SFTTrainer
from transformers import (
    set_seed,
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    AutoConfig,
    BitsAndBytesConfig,
)
from peft import (
    TaskType,
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
    AutoPeftModelForCausalLM,
    PeftConfig
)
# Logging and secrets
from huggingface_hub import login, HfApi, create_repo
from google.colab import userdata
import wandb

In [4]:
import gc

In [5]:
set_seed(42)

In [6]:
wandb_api_key = userdata.get('WANDB_API_KEY')
hf_token = userdata.get('HF_TOKEN')

In [7]:
if hf_token:
    # Log in to Hugging Face
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Hugging Face token not found in notebook secrets.")


Successfully logged in to Hugging Face!


In [8]:
if wandb_api_key:
  wandb.login(key=wandb_api_key)
  print("Successfully logged in to WANDB!")
else:
    print("WANDB key not found in notebook secrets.")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: stevenqcly (stevenqcly-the-university-of-texas-at-dallas) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged in to WANDB!


In [9]:
def free_gpu_memory():
    """
    Frees up GPU memory after CUDA out-of-memory error in Colab.

    This function performs the following steps:
    1. Deletes all PyTorch objects to clear references.
    2. Calls garbage collection to remove unreferenced objects from memory.
    3. Uses torch.cuda.empty_cache() to release cached GPU memory.
    4. Waits for a moment to ensure memory is fully released.
    """
    try:
        # Delete all torch tensors to free up memory
        for obj in list(locals().values()):
            if torch.is_tensor(obj):
                del obj

        # Collect garbage to release any remaining unused memory
        gc.collect()

        # Empty the CUDA cache to release GPU memory
        torch.cuda.empty_cache()

        # Adding a small delay to allow memory to be fully released
        time.sleep(2)

        print("GPU memory has been freed.")
    except Exception as e:
        print(f"Error while freeing GPU memory: {e}")

In [10]:
free_gpu_memory()

Error while freeing GPU memory: name 'time' is not defined


In [11]:
from pathlib import Path
# Determine the storage location based on the execution environment
# If running on Google Colab, use Google Drive as storage
if 'google.colab' in str(get_ipython()):
    from google.colab import drive  # Import Google Drive mounting utility
    drive.mount('/content/drive')  # Mount Google Drive

    # Set base folder path for storing data on Google Drive
    base_folder= Path('/content/drive/My Drive/Fall_2025_Notes/NLP/HW 6')
    project_folder = Path('/content/drive/My Drive/Fall_2025_Notes/NLP/HW 6')
# If running locally, specify a different path
else:
    # Set base folder path for storing data on local machine
    base_folder= Path('/home/')
    project_folder = Path('/home/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# Students can set the data_folder and model_folder paths based on their base_folder setup.
data_folder = base_folder/'datasets'
model_folder = project_folder/'models'
model_folder.mkdir(exist_ok=True, parents = True)
data_folder.mkdir(exist_ok=True, parents = True)

In [13]:
#Now import the2  sets
train_path = data_folder / "train.csv"
test_path  = data_folder / "test.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

train_df.head()


,ID,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,2017-21441,“Worry is a down payment on a problem you may ...,0,1,0,0,0,0,1,0,0,0,1
1,2017-31535,Whatever you decide to do make sure it makes y...,0,0,0,0,1,1,1,0,0,0,0
2,2017-21068,@Max_Kellerman it also helps that the majorit...,1,0,1,0,1,0,1,0,0,0,0
3,2017-31436,Accept the challenges so that you can literall...,0,0,0,0,1,0,1,0,0,0,0
4,2017-22195,My roommate: it's okay that we can't spell bec...,1,0,1,0,0,0,0,0,0,0,0


In [14]:

TEXT_COL = "Tweet"
ID_COL   = "ID"

label_cols = [c for c in train_df.columns if c not in [ID_COL, TEXT_COL]]
label_cols


['anger',
 'anticipation',
 'disgust',
 'fear',
 'joy',
 'love',
 'optimism',
 'pessimism',
 'sadness',
 'surprise',
 'trust']

In [15]:
def build_label_string(row):
    labels = [lbl for lbl in label_cols if row[lbl] == 1]
    return json.dumps(labels)

df = train_df[[TEXT_COL] + label_cols].copy()
df["label"] = df.apply(build_label_string, axis=1)
df = df.rename(columns={TEXT_COL: "text"})
df = df[["text", "label"]]

df.head()


,text,label
0,“Worry is a down payment on a problem you may ...,"[""anticipation"", ""optimism"", ""trust""]"
1,Whatever you decide to do make sure it makes y...,"[""joy"", ""love"", ""optimism""]"
2,@Max_Kellerman it also helps that the majorit...,"[""anger"", ""disgust"", ""joy"", ""optimism""]"
3,Accept the challenges so that you can literall...,"[""joy"", ""optimism""]"
4,My roommate: it's okay that we can't spell bec...,"[""anger"", ""disgust""]"


In [16]:
#Test/train split

SEED = 42

# Step 1: split out test (15%)
train_val_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=SEED,
    shuffle=True
)

# Step 2: split remaining into train / validation
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.15 / 0.85,
    random_state=SEED,
    shuffle=True
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))


Train size: 5406
Validation size: 1159
Test size: 1159


In [17]:
#Conver to hf
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df.reset_index(drop=True))

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

dataset


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5406
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1159
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1159
    })
})

In [18]:
label_names = label_cols

def format_prompt_completion(example):
    prompt = (
        f"Classify the TEXT by selecting all applicable labels from the following list: {label_names}. "
        f"### TEXT: {example['text'].strip()} ### LABEL:"
    )
    completion = f" {example['label'].strip()}"
    return {"prompt": prompt, "completion": completion}

data_subset_completion = dataset.map(
    format_prompt_completion,
    remove_columns=["text", "label"]
)

def to_text(example):
    example["text"] = example["prompt"] + example["completion"]
    return example

data_subset_completion = data_subset_completion.map(to_text)


Map:   0%|          | 0/5406 [00:00<?, ? examples/s]

Map:   0%|          | 0/1159 [00:00<?, ? examples/s]

Map:   0%|          | 0/1159 [00:00<?, ? examples/s]

Map:   0%|          | 0/5406 [00:00<?, ? examples/s]

Map:   0%|          | 0/1159 [00:00<?, ? examples/s]

Map:   0%|          | 0/1159 [00:00<?, ? examples/s]

In [19]:
#tokenizer
model_name = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [20]:
data_subset_completion = data_subset_completion.map(
    lambda ex: {"keep": len(tokenizer.encode(ex["prompt"])) <= 500}
)
data_subset_completion = data_subset_completion.filter(lambda x: x["keep"])
data_subset_completion = data_subset_completion.remove_columns(["keep"])

Map:   0%|          | 0/5406 [00:00<?, ? examples/s]

Map:   0%|          | 0/1159 [00:00<?, ? examples/s]

Map:   0%|          | 0/1159 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5406 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1159 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1159 [00:00<?, ? examples/s]

In [21]:
print(data_subset_completion["train"][0]["text"][:500])

Classify the TEXT by selecting all applicable labels from the following list: ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']. ### TEXT: im literally shaking bc im nervous and bc its fucking cold oh how i love life ### LABEL: ["anger", "disgust", "fear", "sadness"]


In [22]:
#Load model
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    dtype = torch.bfloat16
else:
    dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=dtype,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [23]:
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
)

In [24]:
# Determine torch data type
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
    torch_data_type = torch.bfloat16
else:
    torch_data_type = torch.float16

In [25]:
# Define the directory where model checkpoints will be saved

model_folder = Path("/content/models/gemma_qlora_lmh")
# Create the directory if it doesn't exist
model_folder = Path("/content/models/gemma_qlora_lmh") #Going to save the checkpoints to the environemnt instead of my physical drive
run_name= 'stack_exp_lmh_gemma_base'

use_fp16 = torch_data_type == torch.float16
use_bf16 = torch_data_type == torch.bfloat16

# Configure training parameters
training_args = SFTConfig(
    seed = 42,
    dataset_text_field="text",
    max_length = 1024,
    packing = False,
    completion_only_loss=True,

    # Training-specific configurations
    num_train_epochs=2,  # Total number of training epochs
    per_device_train_batch_size=16, # Number of samples per training batch for each device
    per_device_eval_batch_size=16,  # Number of samples per evaluation batch for each device
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant":False},
    torch_empty_cache_steps=20,
    weight_decay=0.0,  # Apply L2 regularization to prevent overfitting
    learning_rate=1e-5,  # Step size for the optimizer during training
    optim='adamw_torch',  # Optimizer,

    # Checkpoint saving and model evaluation settings
    output_dir=str(model_folder),  # Directory to save model checkpoints
    eval_strategy='steps',  # Evaluate model at specified step intervals
    eval_steps=20,  # Perform evaluation every 10 training steps
    save_strategy="steps",  # Save model checkpoint at specified step intervals
    save_steps=20,  # Save a model checkpoint every 10 training steps
    load_best_model_at_end=True,  # Reload the best model at the end of training
    save_total_limit=2,  # Retain only the best and the most recent model checkpoints
    # Use 'accuracy' as the metric to determine the best model
    metric_for_best_model="eval_loss",
    greater_is_better=False,  # A model is 'better' if its accuracy is higher


    # Experiment logging configurations (commented out in this example)
    logging_strategy='steps',
    logging_steps=20,
    report_to='wandb',  # Log metrics and results to Weights & Biases platform
    run_name= run_name,  # Experiment name for Weights & Biases

    # Precision settings determined based on GPU capability
    fp16=use_fp16 ,  # Set True if torch_data_type is torch.float16
    bf16=use_bf16,  # Set True if torch_data_type is torch.bfloat16
    tf32=False,  # Disable tf32 unless you want to use Ampere specific optimization
)


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [26]:
#I can't get the TRL datacollator for complettion only LM to work. No matter what I do it won't load. So I'm going to use a workaround that still collates

In [27]:
#Below is the custom class i'm using because the TRL data collator doesn't work :(
from dataclasses import dataclass
from typing import List, Dict, Any

@dataclass
class DataCollatorForCompletionOnlyLM:
    """
    TRL-compatibility collator: implements completion-only loss masking.

    Robust behavior:
    - Finds the completion boundary by searching decoded text for `response_template`
      (handles whitespace/newline differences)
    - Masks all tokens up to and including the boundary
    - Masks padding tokens so they don't contribute to loss
    - Ignores TRL extra keys (e.g., 'completion_mask') that break tokenizer.pad()
    """
    tokenizer: Any
    response_template: str
    max_length: int = 1024

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Case 1: Raw text examples
        if "input_ids" not in features[0]:
            texts = [f["text"] for f in features]
            batch = self.tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )
        # Case 2: Already tokenized examples (TRL may include extra keys like completion_mask)
        else:
            safe_features = []
            for f in features:
                item = {"input_ids": f["input_ids"]}
                if "attention_mask" in f:
                    item["attention_mask"] = f["attention_mask"]
                safe_features.append(item)

            batch = self.tokenizer.pad(
                safe_features,
                padding=True,
                return_tensors="pt",
            )

        input_ids = batch["input_ids"]
        attention_mask = batch.get("attention_mask", None)

        labels = input_ids.clone()

        for i in range(input_ids.size(0)):
            seq_ids = input_ids[i].tolist()

            # Decode with special tokens so the boundary is visible exactly as in the sequence
            decoded = self.tokenizer.decode(seq_ids, skip_special_tokens=False)

            # Find the last occurrence of the boundary string
            idx = decoded.rfind(self.response_template)
            if idx == -1:
                print(f"[WARN] response_template not found in decoded text for sample {i}. No masking applied.")
                continue

            # Compute token index where completion starts:
            # tokenize the prefix up to end of boundary and count tokens
            boundary_end = idx + len(self.response_template)
            prefix_text = decoded[:boundary_end]

            # Convert prefix text back to token ids and use its length as cutoff
            prefix_ids = self.tokenizer.encode(prefix_text, add_special_tokens=False)
            cutoff = len(prefix_ids)

            labels[i, :cutoff] = -100

        # Mask padding tokens
        if attention_mask is not None:
            labels = labels.masked_fill(attention_mask == 0, -100)

        out = {"input_ids": input_ids, "labels": labels}
        if attention_mask is not None:
            out["attention_mask"] = attention_mask
        return out


In [28]:
response_template = "### LABEL:"

data_collator = DataCollatorForCompletionOnlyLM(
    tokenizer=tokenizer,
    response_template=response_template,
    max_length=training_args.max_length
)

In [29]:
train_filtered = data_subset_completion["train"]
valid_filtered = data_subset_completion["validation"]

if training_args.gradient_checkpointing:
    model.config.use_cache = False  # Disable caching for compatibility

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_filtered,
    eval_dataset=valid_filtered,
    peft_config=peft_config,
    data_collator=data_collator,
    processing_class=tokenizer
)

Adding EOS to train dataset:   0%|          | 0/5406 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5406 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5406 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1159 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1159 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1159 [00:00<?, ? examples/s]

In [30]:
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))

print(batch['input_ids'][0][0:5])
print(tokenizer.decode(batch['input_ids'][0][0:5]))
print(batch['labels'][0][0:5])

print(len(batch['input_ids'][0]))
print(len(batch['labels'][0]))

print(f"\nINPUTS")
print(f"{'-'*80}")
print(batch['input_ids'][0][99:114])
print(f"\nLABELS")
print(f"{'-'*80}")
print(batch['labels'][0][99:114])
print(f"\nTokens")
print(f"{'-'*80}")
print(tokenizer.decode(batch['input_ids'][0][99:114]))


You're using a GemmaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


tensor([1, 1, 1, 1, 1], device='cuda:0')
<eos><eos><eos><eos><eos>
tensor([-100, -100, -100, -100, -100], device='cuda:0')
108
108

INPUTS
--------------------------------------------------------------------------------
tensor([  3375,    824,    664,  12870,    824,    664, 122384, 235262,   4437],
       device='cuda:0')

LABELS
--------------------------------------------------------------------------------
tensor([  3375,    824,    664,  12870,    824,    664, 122384, 235262,   4437],
       device='cuda:0')

Tokens
--------------------------------------------------------------------------------
joy", "love", "optimism"]


In [31]:
def verify_loss_masking(tokenizer, trainer, num_samples=3):
    """
    Verify which tokens contribute to loss (labels != -100)
    for a few samples from the training dataloader.
    """
    dataloader = trainer.get_train_dataloader()
    batch = next(iter(dataloader))

    for i in range(min(num_samples, len(batch["input_ids"]))):
        input_ids = batch["input_ids"][i]
        labels = batch["labels"][i]

        print(f"\n{'='*80}")
        print(f"Sample {i+1}")
        print(f"{'='*80}")

        # Decode full sequence for reference
        full_text = tokenizer.decode(input_ids, skip_special_tokens=False)
        pprint(f"\nFull text:\n{full_text}")

        # Identify tokens used for loss
        loss_token_indices = (labels != -100).nonzero(as_tuple=True)[0]

        if len(loss_token_indices) == 0:
            print("All tokens masked — no loss will be calculated.")
            continue

        print(f"\nTokens contributing to loss ({len(loss_token_indices)} total):")
        print(f"{'-'*80}")
        print(f"{'Index':<8} {'Token ID':<10} {'Token Text'}")
        print(f"{'-'*80}")

        for idx in loss_token_indices.tolist():
            token_id = input_ids[idx].item()
            token_text = tokenizer.decode([token_id], skip_special_tokens=False)
            print(f"{idx:<8} {token_id:<10} {repr(token_text)}")

        print(f"{'-'*80}")
        print(f"Percentage of tokens used for loss: {len(loss_token_indices)/len(labels)*100:.2f}%")



In [32]:
from pprint import pprint
# Call this after creating your trainer
verify_loss_masking(tokenizer, trainer, num_samples=2)


Sample 1
('\n'
 'Full text:\n'
 '<eos><eos><eos><eos><eos><eos><eos><eos><eos><eos><bos>Classify the TEXT by '
 "selecting all applicable labels from the following list: ['anger', "
 "'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', "
 "'sadness', 'surprise', 'trust']. ### TEXT: @LethbridgeCity I love how "
 'playful you guys are becoming -- both on- and offline! 🚗 🚕 🚙 #beepbeep ### '
 'LABEL: ["joy", "love", "optimism"]')

Tokens contributing to loss (10 total):
--------------------------------------------------------------------------------
Index    Token ID   Token Text
--------------------------------------------------------------------------------
98       10890      ' ["'
99       3375       'joy'
100      824        '",'
101      664        ' "'
102      12870      'love'
103      824        '",'
104      664        ' "'
105      122384     'optimis'
106      235262     'm'
107      4437       '"]'
------------------------------------------------------

In [33]:
%env WANDB_PROJECT=multilabel_PartBNotebook1_2025


env: WANDB_PROJECT=multilabel_PartBNotebook1_2025


In [34]:
try:
    trainer.train()
except RuntimeError as e:
    if 'CUDA out of memory' in str(e):
        print("CUDA out of memory error detected. Freeing GPU memory.")
        clear_gpu()   # or free_gpu_memory() if that’s what your notebook defines
    else:
        raise e


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,0.855100,0.520731,1.730762,58673.000000,0.818400
40,0.462200,0.447602,1.690972,117785.000000,0.837579
60,0.412700,0.429730,1.658098,176503.000000,0.841418
80,0.420400,0.408637,1.650292,235625.000000,0.847461
100,0.404900,0.404687,1.614583,294811.000000,0.848466
120,0.374100,0.401188,1.624790,353499.000000,0.847560
140,0.390600,0.393133,1.630383,412395.000000,0.849722
160,0.389700,0.393900,1.633156,471075.000000,0.847376
180,0.367200,0.386607,1.609070,529731.000000,0.851851
200,0.366400,0.388350,1.624645,589065.000000,0.852693


# Model Checkpoints

In [35]:
best_model_checkpoint_step = trainer.state.best_model_checkpoint.split('-')[-1]

In [36]:
best_model_checkpoint_step

'320'

In [37]:
checkpoint = str(model_folder/f'checkpoint-{best_model_checkpoint_step}')
checkpoint

'/content/models/gemma_qlora_lmh/checkpoint-320'

In [38]:
#Savingg my best checkpoint so that i don't lose it and i can use in the next notebook
best_ckpt = trainer.state.best_model_checkpoint
print("best_ckpt:", best_ckpt)

final_adapter_dir = project_folder / "models" / "best_adapter_export"
final_adapter_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(final_adapter_dir)   # saves adapter_* files for PEFT models
tokenizer.save_pretrained(final_adapter_dir)

print("Saved adapter to:", final_adapter_dir)


best_ckpt: /content/models/gemma_qlora_lmh/checkpoint-320
Saved adapter to: /content/drive/My Drive/Fall_2025_Notes/NLP/HW 6/models/best_adapter_export


In [39]:
wandb.finish()

eval/entropy,█▆▄▃▁▂▂▂▁▂▁▂▁▂▂▂
eval/loss,█▄▄▃▂▂▂▂▁▂▁▁▁▁▁▁
eval/mean_token_accuracy,▁▅▆▇▇▇▇▇██▇█████
eval/num_tokens,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
eval/runtime,▅▄▁▁▁▂▇█▂▂▃▁▃▂▂▃
eval/samples_per_second,▄▅███▇▂▁▇▇▆█▆▇▇▆
eval/steps_per_second,▄▅███▇▂▁▇▇▆█▆▇▇▆
train/entropy,█▇▄▄▃▁▂▂▃▂▂▁▁▁▂▁
train/epoch,▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇███
+5,...
